# 04 — Structured Streaming, UDFs & AQE

In [ ]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


## Structured Streaming

In [ ]:
import time
stream_df = spark.readStream.format('rate').option('rowsPerSecond', 20).load()
agg = stream_df.withColumn('bucket', (F.col('value') % 5).cast('string'))                .groupBy('bucket').count()
query = agg.writeStream.outputMode('complete').format('memory').queryName('rate_agg').start()
time.sleep(5)
spark.sql('SELECT * FROM rate_agg ORDER BY bucket').show()
query.stop()


## UDF & Pandas UDF

In [ ]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

@udf(returnType=StringType())
def classify_salary(salary):
    if salary is None: return 'unknown'
    if salary > 90000: return 'high'
    if salary > 70000: return 'mid'
    return 'low'

@pandas_udf(DoubleType())
def tax_estimate(salary: pd.Series) -> pd.Series:
    return salary * 0.3

schema = StructType([StructField('id',IntegerType()),StructField('name',StringType()),
                     StructField('department',StringType()),StructField('salary',DoubleType()),StructField('country',StringType())])
employees = spark.createDataFrame([(1,'Alice','Engineering',95000.0,'US'),(2,'Bob','Marketing',72000.0,'UK'),
    (3,'Carol','Engineering',88000.0,'US'),(4,'Dave','HR',65000.0,'DE')], schema)
employees.withColumn('band', classify_salary('salary'))          .withColumn('tax',  tax_estimate('salary')).show()


## Adaptive Query Execution (AQE)

In [ ]:
print('AQE enabled:', spark.conf.get('spark.sql.adaptive.enabled'))
skewed = spark.range(200_000)     .withColumn('key', F.when(F.col('id') < 190_000, F.lit(1)).otherwise((F.col('id') % 50).cast('int')))     .withColumn('val', F.rand())
other = spark.range(50).withColumn('key', F.col('id').cast('int')).withColumn('info', F.lit('x'))
joined = skewed.join(other, 'key').groupBy('key').count()
joined.explain(True)
